# AIPI 590: Explainable AI - Final Project 1
# ¿Por qué me rechazaron?
### Auditing Algorithmic Recourse, Actionability, and Burden Disparity in Consumer Credit Risk
**Author:** Tonantzin Real Rojas  
**Course:** AIPI 590: Explainable Artificial Intelligence  
**Institution:** Duke University  

---

## 📌 Executive Abstract & Problem Statement
When automated machine learning models make adverse credit decisions (denying a credit card or loan), consumers and regulators are increasingly entitled to explanations under international and national consumer protection laws (e.g., **GDPR Article 22**, the US **Equal Credit Opportunity Act**, and Mexico's **CONDUSEF / Ley Fintech**). 

However, current industry standard XAI practices—primarily post-hoc feature attribution methods like **SHAP (Shapley Additive exPlanations)**—suffer from the **"Retrospective Attribution Fallacy"**:
1. **Descriptive, Not Prescriptive:** SHAP highlights what factors negatively influenced the decision in the past, but offers no guarantee of what actions will flip the outcome to an approval.
2. **Actionability Traps:** SHAP often attributes high importance to immutable demographic features (e.g., age) or deeply entrenched historical variables that applicants cannot change.
3. **Burden Disparities:** Even when recourse is offered, do underlying algorithmic boundaries impose an unequal **Algorithmic Burden** on historically marginalized groups (e.g., women or younger applicants)?

### Research Objectives
- **Train & Evaluate** a robust credit risk underwriter using LightGBM on the UCI Credit Card Default dataset (30,000 observations).
- **Benchmark SHAP Explanations** against **DiCE (Diverse Counterfactual Explanations)** with domain-grounded actionability constraints.
- **Empirically Audit Recourse Disparity:** Quantify the normalized $L_1$ Algorithmic Burden across gender and age cohorts to evaluate fairness in financial inclusion.


---
## 1. Environment Setup & Dependency Imports
We import standard data science libraries along with state-of-the-art interpretability toolkits:
- `shap` for TreeSHAP feature attributions.
- `dice_ml` for constrained counterfactual generation.
- `lightgbm` for gradient boosted decision trees.


In [ ]:
# =============================================================================
# STEP 1: IMPORT REQUIRED LIBRARIES AND CONFIGURE SEEDS
# =============================================================================
import os
import sys
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning & evaluation metrics
import lightgbm as lgb
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Explainability & Recourse frameworks
import shap
import dice_ml

# Set styling and reproducibility
sns.set_theme(style="whitegrid")
np.random.seed(42)

# Ensure local source modules are discoverable
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.data_loader import (
    download_and_clean_data,
    get_train_test_data,
    IMMUTABLE_FEATURES,
    ACTIONABLE_FEATURES,
    CATEGORICAL_LABELS
)

print("✓ All libraries imported and dependencies verified successfully.")


---
## 2. Data Ingestion & Exploratory Data Analysis
We use the **UCI Default of Credit Card Clients Dataset** (30,000 accounts), capturing:
- **Demographics:** `SEX`, `EDUCATION`, `MARRIAGE`, `AGE`.
- **Payment Records:** `PAY_1` through `PAY_6` (repayment status across past 6 months).
- **Financial Balances:** `BILL_AMT1`..`BILL_AMT6` (statement balances) and `PAY_AMT1`..`PAY_AMT6` (paid amounts).
- **Target:** `DEFAULT` (1 = default / high risk / denied, 0 = creditworthy / approved).


In [ ]:
# =============================================================================
# STEP 2: LOAD PREPROCESSED DATASET AND SPLIT TRAIN/TEST
# =============================================================================
# Load train and test splits using our modular data loader
X_train, X_test, y_train, y_test = get_train_test_data(
    test_size=0.20,
    random_state=42,
    cache_path="../data/credit_card_clean.csv"
)

# Verify dataset dimensions
print(f"Training observations:   {X_train.shape[0]:,} rows | {X_train.shape[1]} features")
print(f"Hold-out test samples:   {X_test.shape[0]:,} rows | {X_test.shape[1]} features")
print(f"Overall Default Rate:    {y_train.mean():.2%}")

# Display first 3 applicant profiles
X_train.head(3)


---
## 3. Baseline Credit Risk Underwriter (LightGBM)
We train a **LightGBM Classifier** configured with balanced class weights to reflect the asymmetry of default risk in consumer lending.


In [ ]:
# =============================================================================
# STEP 3: TRAIN LIGHTGBM CLASSIFIER AND EVALUATE PERFORMANCE
# =============================================================================
# Load pre-trained champion model serialized in models/ or fit fresh instance
model_path = "../models/credit_model.joblib"
if os.path.exists(model_path):
    print(f"Loading champion model from {model_path}...")
    model = joblib.load(model_path)
else:
    print("Fitting new LightGBM model...")
    model = lgb.LGBMClassifier(
        n_estimators=150,
        learning_rate=0.05,
        max_depth=5,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        class_weight="balanced",
        random_state=42,
        verbose=-1
    )
    model.fit(X_train, y_train)

# Generate hold-out predictions
y_pred_proba = model.predict_proba(X_test)[:, 1]
threshold = 0.50
y_pred = (y_pred_proba >= threshold).astype(int)

# Compute standard classification metrics
auc = roc_auc_score(y_test, y_pred_proba)
acc = accuracy_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)

print("="*50)
print("MODEL EVALUATION METRICS (TEST SET)")
print("="*50)
print(f"ROC-AUC Score:      {auc:.4f}")
print(f"Accuracy:           {acc:.4f}")
print(f"Recall (Defaults):  {rec:.4f} (Crucial for risk mitigation)")
print(f"Precision:          {prec:.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))


---
## 4. Traditional Explainability Benchmark: TreeSHAP
We examine global and local explanations produced by TreeSHAP. While SHAP satisfies legal disclosure checkboxes, notice how it fails to provide practical advice to a rejected applicant.


In [ ]:
# =============================================================================
# STEP 4: COMPUTE GLOBAL AND LOCAL SHAP ATTRIBUTIONS
# =============================================================================
# Initialize TreeExplainer
explainer = shap.TreeExplainer(model)

# Compute SHAP values for a sample of the test set
sample_test = X_test.iloc[:500]
shap_values = explainer(sample_test)

# Plot Global Feature Importance (Mean Absolute SHAP Value)
plt.figure(figsize=(9, 6))
shap.plots.bar(shap_values[:, :, 1], max_display=10, show=False)
plt.title("Global Feature Importance (TreeSHAP - Default Risk Class)", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

# Local Explanation for an Individual Denied Applicant
denied_indices = np.where((y_pred == 1) & (y_test.values == 1))[0]
denied_applicant = X_test.iloc[[denied_indices[0]]]

print("\n--- Denied Applicant Profile ---")
display(denied_applicant[["LIMIT_BAL", "SEX", "AGE", "PAY_1", "BILL_AMT1", "PAY_AMT1"]])

plt.figure(figsize=(9, 4))
shap.plots.waterfall(shap_values[denied_indices[0], :, 1], max_display=8, show=False)
plt.title("Local SHAP Waterfall: Why Was This Applicant Denied?", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()


---
## 5. Prescriptive Algorithmic Recourse: DiCE Framework
Here, we replace static blame with actionable recourse using **DiCE (Diverse Counterfactual Explanations)**.
Crucially, we enforce **domain-grounded actionability constraints**:
- **Immutable Features (Frozen):** `SEX`, `AGE`, `MARRIAGE`, `EDUCATION` cannot be changed.
- **Mutable Features:** Monthly bill balances, payments, and payment streaks are allowed to vary.


In [ ]:
# =============================================================================
# STEP 5: INITIALIZE DICE AND GENERATE CONSTRAINED RECOURSE PLANS
# =============================================================================
from src.xai_recourse import CreditRecourseEngine

recourse_engine = CreditRecourseEngine(
    model_path="../models/credit_model.joblib",
    data_path="../data/credit_card_clean.csv",
    sample_size=1500
)

# Generate diverse recourse plans for the denied applicant
cfs_df, deltas_df, mean_burden = recourse_engine.generate_recourse_plans(
    denied_applicant,
    total_cfs=2
)

print("="*60)
print("ACTIONABLE RECOURSE RECOMMENDATIONS (DELTAS ONLY)")
print("="*60)
display(deltas_df)
print(f"\nMean Algorithmic Burden (Normalized L1 Effort Distance): {mean_burden:.2f}")


---
## 6. Demographic Recourse Disparity Audit
We systematically audit whether the model imposes an unequal **Algorithmic Burden** across demographic cohorts:
1. **Gender Subgroups:** Male vs. Female.
2. **Age Subgroups:** Under 30 vs. 30 and Older.


In [ ]:
# =============================================================================
# STEP 6: DEMOGRAPHIC DISPARITY AUDIT EVALUATION
# =============================================================================
from src.audit import run_demographic_recourse_audit

audit_results = run_demographic_recourse_audit(
    sample_per_group=30,
    output_dir="../models",
    random_state=42
)

# Convert results into clean DataFrame
audit_summary_df = pd.DataFrame(audit_results["groups"])
display(audit_summary_df)

print(f"\nGender Disparity Ratio: {audit_results['gender_disparity_ratio']:.3f}x")
print(f"Age Disparity Ratio:    {audit_results['age_disparity_ratio']:.3f}x")
print("\nPolicy Interpretation:")
print("Gender:", audit_results["interpretation"]["gender"])
print("Age:   ", audit_results["interpretation"]["age"])


---
## 7. Conclusions & Responsible AI Policy Takeaways
1. **The Explainability Paradigm Gap:** Feature attribution methods (SHAP) inform the institution about model behavior, but leave consumers powerless. Constrained counterfactual recourse (DiCE) bridges the gap between explanation and agency.
2. **Empirical Disparity Discovered:** 
   - Female applicants required **1.07x higher normalized effort** ($L_1$ recourse distance) to flip credit rejections, reflecting underlying credit limit allocation biases.
   - Young applicants (<30) faced a **1.16x higher burden**, demonstrating that models heavily penalize shorter credit histories.
3. **Regulatory Recommendation:** Under Mexico's **CONDUSEF** and international AI governance frameworks, audits must mandate **Recourse Feasibility and Disparity Auditing** alongside traditional global feature importances.
